## Graph data

## Setup

In [1]:
!pip install pandas plotly numpy


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Import dati

In [1]:
import pandas as pd
import json
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Carica i file CSV
perf_df = pd.read_csv('stopwatch_runs.csv')
fake_df = pd.read_csv('fake_dataset.csv')

# Analizza i tipi di colonna dal dataset
print("Tipi di colonne in fake_dataset.csv:")
print(fake_df.dtypes)
print("\nValori unici per f4:", fake_df['f4'].nunique())
print("Valori unici per f5:", fake_df['f5'].nunique())
print("\nDistribuzione f4:", fake_df['f4'].value_counts().index.tolist())
print("Distribuzione f5:", fake_df['f5'].value_counts().index.tolist())

FileNotFoundError: [Errno 2] No such file or directory: 'fake_dataset.csv'

In [ ]:
# Estrai affected_features da strategy_json
perf_df['affected_feature'] = perf_df['strategy_json'].apply(
    lambda x: json.loads(x.replace('""', '"'))['affected_features'][0]
)

# Definisci i tipi di colonna per la legenda
column_types = {
    'f1': 'float64',
    'f2': 'int64',
    'f3': 'categorical (3 classi)',
    'f4': 'categorical (3 classi)',
    'f5': 'int64 (6 classi)',
    'date': 'datetime',
    'target': 'binary'
}

print("Dati estratti:")
print(perf_df[['metodo', 'backend', 'affected_feature', 'stopwatch_sec']].head(20))

Dati estratti:
        metodo backend affected_feature  stopwatch_sec
0   duplicated  PANDAS               f1       0.512010
1   duplicated   SPARK               f1       6.404904
2   duplicated  PANDAS               f2       0.497250
3   duplicated   SPARK               f2       1.795457
4   duplicated  PANDAS               f3       0.500013
5   duplicated   SPARK               f3       1.651567
6   duplicated  PANDAS               f4       0.502125
7   duplicated   SPARK               f4       1.483769
8   duplicated  PANDAS               f5       0.493062
9   duplicated   SPARK               f5       1.464234
10  duplicated  PANDAS             date       0.493285
11  duplicated   SPARK             date       1.437912
12  duplicated  PANDAS           target       0.500428
13  duplicated   SPARK           target       1.438766
14     missing  PANDAS               f1       0.177129
15     missing   SPARK               f1       2.298614
16     missing  PANDAS               f2       0.17

## Grafico MainTests

In [ ]:
# Crea due grafici separati per backend (SPARK e PANDAS)

# Mappa colori per tipo di colonna
color_map = {
    'f1': '#d62728',      # rosso (float)
    'f2': '#f7b801',      # giallo (int)
    'f3': '#2ca02c',      # verde (categorical)
    'f4': '#1f77b4',      # blu (categorical)
    'f5': '#17becf',      # ciano (int)
    'date': '#9467bd',    # viola (datetime)
    'target': '#8c564b'   # marrone (binary)
}

# Crea figura con 2 sottografici
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=("Backend: SPARK", "Backend: PANDAS"),
    vertical_spacing=0.12,
    specs=[[{"secondary_y": False}], [{"secondary_y": False}]]
)

metodi = ['outlier', 'duplicated', 'missing', 'noise']
affected_features = ['f1', 'f2', 'f3', 'f4', 'f5', 'date', 'target']
backends = ['SPARK', 'PANDAS']

# Per ogni backend
for row_idx, backend in enumerate(backends, start=1):
    backend_data = perf_df[perf_df['backend'] == backend]
    
    # Aggiungi barre per ogni colonna
    for feature in affected_features:
        feature_data = backend_data[backend_data['affected_feature'] == feature]
        
        # Calcola i valori medi per ogni metodo
        metodo_values = []
        for metodo in metodi:
            metodo_data = feature_data[feature_data['metodo'] == metodo]['stopwatch_sec'].values
            metodo_values.append(np.mean(metodo_data) if len(metodo_data) > 0 else 0)
        
        fig.add_trace(
            go.Bar(
                x=metodi,
                y=metodo_values,
                name=f'{feature} ({column_types[feature]})',
                marker=dict(color=color_map[feature]),
                text=[f'{v:.2f}' for v in metodo_values],
                textposition='auto',
                legendgroup=feature,
                showlegend=(row_idx == 1),  # Mostra legenda solo per il primo grafico
            ),
            row=row_idx, col=1
        )

# Calcola il massimo valore da tutti i dati per fissare la scala
max_value = perf_df['stopwatch_sec'].max()
y_max = max_value * 1.1  # Aggiunge il 10% di spazio sopra

fig.update_xaxes(title_text="Metodo", row=2, col=1)
fig.update_xaxes(title_text="", row=1, col=1)

# Applica la stessa scala Y per entrambi i grafici
fig.update_yaxes(title_text="Tempo (sec)", range=[0, y_max], row=1, col=1)
fig.update_yaxes(title_text="Tempo (sec)", range=[0, y_max], row=2, col=1)

fig.update_layout(
    title='Confronto Performance: SPARK vs PANDAS (1.000.000 records)',
    height=900,
    width=1200,
    font=dict(size=11),
    hovermode='x unified',
    barmode='group'
)

fig.show()

In [ ]:
# Carica il file stress_test.csv
stress_df = pd.read_csv('/home/cava/Documents/Repos/python/pucktrick/performance_test/stress_test.csv')

# Estrai affected_features da strategy_json
stress_df['affected_feature'] = stress_df['strategy_json'].apply(
    lambda x: json.loads(x.replace('""', '"'))['affected_features'][0]
)

print("Dati stress test:")
print(stress_df[['metodo', 'backend', 'iterazione', 'num_righe', 'stopwatch_sec_iter', 'stopwatch_sec_tot']].head(20))
print("\nMetodi unici:", stress_df['metodo'].unique())
print("Backend unici:", stress_df['backend'].unique())

Dati stress test:
     metodo backend  iterazione  num_righe  stopwatch_sec_iter  \
0  outliers   SPARK           1    1000000            3.584395   
1  outliers   SPARK           2    2000000            3.042496   
2  outliers   SPARK           3    4000000            3.510963   
3  outliers   SPARK           4    8000000            5.853660   
4  outliers   SPARK           5   16000000           11.021906   
5  outliers   SPARK           6   32000000           20.675027   
6  outliers   SPARK           7   64000000           41.238236   
7  outliers   SPARK           8  128000000           84.641408   

   stopwatch_sec_tot  
0           3.584395  
1           6.626892  
2          10.137855  
3          15.991515  
4          27.013421  
5          47.688449  
6          88.926685  
7         173.568093  

Metodi unici: ['outliers']
Backend unici: ['SPARK']


In [ ]:
# Grafico stress test: Tempo vs Numero di righe per outliers

# Raggruppa per backend
backends_stress = stress_df['backend'].unique()

if len(backends_stress) == 1:
    # Se c'è un solo backend, crea un unico grafico
    backend = backends_stress[0]
    backend_data = stress_df[stress_df['backend'] == backend].sort_values('num_righe')
    
    fig_stress = go.Figure()
    
    fig_stress.add_trace(go.Scatter(
        x=backend_data['num_righe'],
        y=backend_data['stopwatch_sec_iter'],
        mode='lines+markers',
        name='Tempo per iterazione',
        line=dict(color='#1f77b4', width=3),
        marker=dict(size=8)
    ))
    
    fig_stress.update_layout(
        title=f'Stress Test: Tempo di calcolo vs Numero di righe ({backend})',
        xaxis_title='Numero di righe',
        yaxis_title='Tempo (secondi)',
        height=600,
        width=1000,
        font=dict(size=12),
        hovermode='x unified'
    )
    
else:
    # Se ci sono più backend, crea 2 grafici
    fig_stress = make_subplots(
        rows=1, cols=2,
        subplot_titles=[f'Backend: {b}' for b in sorted(backends_stress)],
        specs=[[{"secondary_y": False}, {"secondary_y": False}]]
    )
    
    for col_idx, backend in enumerate(sorted(backends_stress), start=1):
        backend_data = stress_df[stress_df['backend'] == backend].sort_values('num_righe')
        
        fig_stress.add_trace(
            go.Scatter(
                x=backend_data['num_righe'],
                y=backend_data['stopwatch_sec_iter'],
                mode='lines+markers',
                name=backend,
                line=dict(width=3),
                marker=dict(size=8),
                showlegend=(col_idx == 1)
            ),
            row=1, col=col_idx
        )
        
        fig_stress.update_xaxes(title_text="Numero di righe", row=1, col=col_idx)
        fig_stress.update_yaxes(title_text="Tempo (sec)", row=1, col=col_idx)
    
    fig_stress.update_layout(
        title='Stress Test: Tempo di calcolo vs Numero di righe (outliers)',
        height=500,
        width=1400,
        font=dict(size=11),
        hovermode='x unified'
    )

fig_stress.show()

In [ ]:
# ── Breakeven analysis: carica breakeven_runs.csv ─────────────────────────────
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json

be_df = pd.read_csv('breakeven_runs.csv')

# Filtra righe valide (elapsed >= 0; -2 = skipped per OOM, -1 = errore)
be_df = be_df[be_df['stopwatch_sec'] >= 0].copy()

metodi    = ['duplicated', 'missing', 'noise', 'outlier', 'labels']
COLOR_PD  = '#1f77b4'   # blu  → Pandas
COLOR_SP  = '#d62728'   # rosso → Spark
COLOR_BE  = '#2ca02c'   # verde → linea break-even

# Recupera il num_righe di break-even per ogni metodo (None se non trovato)
be_rows = {}
for m in metodi:
    hit = be_df[(be_df['metodo'] == m) & (be_df['breakeven_reached'] == True)]
    be_rows[m] = int(hit['num_righe'].iloc[0]) if len(hit) > 0 else None

# ── Figura: 5 subplot in griglia 2×3 (ultima cella vuota) ─────────────────────
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[m.capitalize() for m in metodi],
    vertical_spacing=0.18,
    horizontal_spacing=0.10,
)

# Mappa metodo → posizione griglia
positions = {m: (r, c) for m, (r, c) in zip(
    metodi,
    [(1,1),(1,2),(1,3),(2,1),(2,2)]
)}

for metodo in metodi:
    row, col = positions[metodo]
    data_m   = be_df[be_df['metodo'] == metodo].sort_values('num_righe')

    for backend, color, dash in [
        ('PANDAS', COLOR_PD, 'dot'),
        ('SPARK',  COLOR_SP, 'solid'),
    ]:
        d = data_m[data_m['backend'] == backend]
        fig.add_trace(
            go.Scatter(
                x=d['num_righe'],
                y=d['stopwatch_sec'],
                mode='lines+markers',
                name=backend,
                legendgroup=backend,
                showlegend=(metodo == 'duplicated'),   # una sola entry per backend
                line=dict(color=color, width=2.5, dash=dash),
                marker=dict(size=7),
                hovertemplate=(
                    f'<b>{metodo} – {backend}</b><br>'
                    'Righe: %{x:,}<br>'
                    'Tempo: %{y:.2f}s<extra></extra>'
                ),
            ),
            row=row, col=col,
        )

    # ── Linea verticale break-even ─────────────────────────────────────────────
    be_x = be_rows[metodo]
    if be_x is not None:
        fig.add_vline(
            x=be_x,
            line=dict(color=COLOR_BE, width=2, dash='dash'),
            row=row, col=col,
            annotation_text=f'BE @ {be_x/1e6:.0f}M',
            annotation_position='top right',
            annotation_font=dict(color=COLOR_BE, size=10),
        )
    else:
        # Nessun break-even trovato nel range testato
        fig.add_annotation(
            text='Nessun BE trovato',
            xref='x domain', yref='y domain',
            x=0.5, y=0.95,
            showarrow=False,
            font=dict(color='gray', size=10),
            row=row, col=col,
        )

    # Asse X in formato leggibile (milioni)
    fig.update_xaxes(
        title_text='Righe',
        tickformat='.2s',
        row=row, col=col,
    )
    fig.update_yaxes(
        title_text='Tempo (s)',
        row=row, col=col,
    )

fig.update_layout(
    title=dict(
        text='Break-even Spark vs Pandas — per metodo di noise injection',
        font=dict(size=16),
        x=0.5,
    ),
    height=700,
    width=1300,
    font=dict(size=11),
    hovermode='x unified',
    legend=dict(
        title='Backend',
        orientation='v',
        x=0.72, y=0.25,
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
)

fig.show()